# III) Single material optimization


## 1) Geometry

First, we define the geometry, material distribution, and generate the mesh.

In [ ]:
pole_pairs = 6
bundles_per_half_slot = 5

# Generate the corresponding mesh
from utils.geometry import machine_mesh
mesh = machine_mesh(p=pole_pairs, 
                    bundles_per_half_slot=bundles_per_half_slot, 
                    hBundle=0.25e-3
                    )

print(f"Generated mesh with {mesh.nv} nodes, {mesh.ne} elements")

________
## 2) Setup of the physical problem

See notebook [I_Reference_simulation](I_Reference_simulation.ipynb).

### 2.a) Material properties

In [ ]:
mur_iron = 1000           # Relative magnetic permeability of iron core
sigma_copper = 5.8e7      # Copper conductivity (S/m)
Br = 1                    # Remanent flux density of magnets (T)


# Define space-varying material coefficients
from ngsolve import pi
mu0 = 4e-7 * pi
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0 * mur_iron)}, default = 1/mu0)     # reluctivity
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper})                     # conductivity
from utils.physics import magnetization_halbach
Mcplx = mesh.MaterialCF({"rotor" : magnetization_halbach(br = Br, p=pole_pairs)}) # magnetization

### 2.b) Current supply

In [ ]:
freq = 1000 # Electrical frequency (Hz)
Jrms = 10e6 # Current density (A/m²)
phi = 150   # load angle (°), chosen to maximize average torque
winding_type = "distributed" # "distributed" or "concentrated"

from ngsolve import Integrate
from utils.supply import phase_current, winding_arrangement, bundle_arrangement

S_bundle = Integrate(1, mesh.Materials("slot11_bundle0"))
Irms = Jrms * S_bundle
phase = phase_current(I_rms=Irms,  load_angle=phi*pi/180)
winding = winding_arrangement(phase, type = winding_type)
bundles = bundle_arrangement(winding, bundles_per_half_slot=bundles_per_half_slot)

### 2.c) Finite element space

In [ ]:
curve_order = 2
fem_order = curve_order

from ngsolve import H1, Periodic
fes = Periodic( H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   dirichlet =  "shaft|out", 
                   complex = True),  [-1]*7 )
print(f"Number of degrees of freedom of the FE space: {fes.ndof}")

_________
## 3) Optimization of winding material

Too high conductivity leads to high AC losses, while too low conductivity leads to high DC losses. There is an optimum point that can be found.

### 3.a) Exhaustive search

Since there is only one scalar optimization variable (the conductivity value), an exhaustive search is possible.

In [ ]:
# Initialization
from numpy import logspace, log10
conductivity_list = logspace(6,log10(sigma_copper),10)
joule_total_losses_list = []
joule_AC_losses_list = []
joule_DC_losses_list = []

In [ ]:
from utils.physics import solve_magnetoharmonic, current_density, joule_losses
from ngsolve import CoefficientFunction as CF, Norm
from ngsolve.webgui import Draw

scene = Draw(CF(0), fes.mesh, 
             settings = {"Objects" : {"Wireframe" : False}})

for sigma_test in conductivity_list:
    result_AC = solve_magnetoharmonic(fes=fes,
                                   reluctivity=nu,
                                   conductivity=mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_test}),
                                   magnetization=Mcplx,
                                   frequency=freq,
                                   supply=bundles
                                   )
    
    result_DC = solve_magnetoharmonic(fes=fes,
                                   reluctivity=nu,
                                   conductivity=mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_test}),
                                   magnetization=Mcplx,
                                   frequency=freq / 1e6,
                                   supply=bundles
                                   )
    
    j = current_density(result_AC)
    scene.Redraw(Norm(j), result_AC["info"]["fes"].mesh, 
                 settings = {"Objects" : {"Wireframe" : False}})    
    joule_total_losses_list.append(joule_losses(result_AC))
    joule_DC_losses_list.append(joule_losses(result_DC))
    joule_AC_losses_list.append(joule_total_losses_list[-1] - joule_DC_losses_list[-1])
    print(f"n = {len(joule_total_losses_list):>02}/{len(conductivity_list)} | σ = {sigma_test:.2e} S/m | P_tot = {joule_total_losses_list[-1]:.2e} W/m")


In [ ]:
# Selection of optimal conductivity
from numpy import argmin
i_min = argmin(joule_total_losses_list)
sigma_min = conductivity_list[i_min]
joule_total_losses_min = joule_total_losses_list[i_min]

# Display
import matplotlib.pyplot as plt
plt.semilogx(conductivity_list, joule_total_losses_list, 'o--', label = "Total losses")
plt.semilogx(conductivity_list, joule_DC_losses_list, 'o--', label = "DC losses")
plt.semilogx(conductivity_list, joule_AC_losses_list, 'o--', label = "AC losses")
plt.legend()
plt.semilogx([min(conductivity_list),sigma_min,sigma_min], 
             [joule_total_losses_min,joule_total_losses_min,0], 'r:')
plt.xlabel("Conductivity (S/m)")
plt.ylabel("Losses (W/m)")
plt.title(f"Copper losses (min for $\\sigma$ = {sigma_min:.2e} S/m)")
plt.savefig("scenes/optim/single_material/exhaustive_search_sigma.pdf")
plt.grid(); plt.show()

In [ ]:
print(f"Optimized total Joule losses (sigma = {conductivity_list[i_min]:.2e} S/m): {joule_total_losses_min:.2e} W/m")
print(f"Optimized DC Joule losses (sigma = {conductivity_list[i_min]:.2e} S/m): {joule_DC_losses_list[i_min]:.2e} W/m ({joule_DC_losses_list[i_min]/joule_total_losses_min*100:.1f} %)")
print(f"Optimized AC Joule losses (sigma = {conductivity_list[i_min]:.2e} S/m): {joule_AC_losses_list[i_min]:.2e} W/m ({joule_AC_losses_list[i_min]/joule_total_losses_min*100:.1f} %)")

The best conductivity at $f=1kHz$ seems to be $\sigma_{min} = 9.54\times 10^6 S/m$, which is a far less than copper ($5.8\times 10^7 S/m$). To summarize:

We find the following results after an exhaustive search at $f = 1\;\text{kHz}$:
- $\sigma_{opt} = 9.54\times 10^6 S/m$

| Total losses at 1kHz | $P_{single} = 1.35\times 10^4 \; \text{W/m}$ | 100% |
|---|---|------|
| DC losses | $P_{DC-single} = 6.12\times 10^3  \; \text{W/m}$ | 45.4 %   |
| AC losses | $P_{AC-single} = 7.37\times 10^3  \; \text{W/m}$ | 54.6 %   |

For a single variable, we somehow expect to have $50$% of DC losses and $50$% of AC losses at the optimum point. We can be more precise with a gradient search.

____
### 3.b) Gradient descent

A gradient descent is also possible, and scalable with the number of input optimization variables. It can also converge to a more precise value.

#### 3.b.i) Setup of the optimization problem
The keypoint is the definition of the "gradient" that also defines the admissible variations of $\sigma$. In the present case the conductivity is the same in all conductor regions.

In [ ]:
# Problem parameters
sigma0 = sigma_copper         # initial conductivity
sigma_min = sigma_copper/10   # minimum admissible conductivity
sigma_max = sigma_copper      # maximum admissible conductivity

# Conductivity space
from ngsolve import L2, GridFunction
fes_sigma = L2(mesh, definedon = "slot(.*)_bundle.*")
x0 = GridFunction(fes_sigma)
x0.Set(sigma0)

In [ ]:
# Definition of the functions of interest

def state_function(conductivity):
    """ Returns the solution of the magnetoharmonic problem for a given conductivity value """
    result = solve_magnetoharmonic(fes=fes,
                                   reluctivity=nu,
                                   conductivity=conductivity,
                                   magnetization=Mcplx,
                                   frequency=freq,
                                   supply=bundles
                                   )
    return result

def objective_function(result):
    """ Total Joule losses to minimize """
    return joule_losses(result)


In [ ]:
# Algorithm parameters

from copy import copy
from numpy import sign

def descent(grad):
    """ Extract descent direction from the gradient """
    descent = copy(grad)
    descent.vec.data = - 1e7 * sign(grad.vec)
    return descent

# Derivative
from utils.optimization import d_joule_losses
from ngsolve import InnerProduct

def grad_joule_losses_winding(result):
    """ Compute the gradient of the objective function w.r.t the conductivity value of the whole winding """
    dx = GridFunction(result["info"]["conductivity"].space)
    dx.Set(1)
    df = d_joule_losses(result, dx) # directional derivative of the objective function in the direction dx
    # we want now to provide a conductivity field representing the steepest admissible ascent direction (we call this the "gradient")
    dfdx = InnerProduct(df.vec, dx.vec) # derivative of f in the direction unit perturbation dx
    grad = copy(dx)
    grad.vec.data = dfdx * dx.vec  # dx is an admissible unit perturbation of the conductivity
    return grad

### 3.b.ii) Optimization loop

In [ ]:
from utils.optimization import gradient_descent

results_optim_single_material = gradient_descent(state=state_function,
                                 objective = objective_function,
                                 d_objective = grad_joule_losses_winding,
                                 x0 = x0,
                                 x_min = sigma_min,
                                 x_max = sigma_max,
                                 descent = descent)

### 3.b.iii) Optimization results

In [ ]:
# Display the optimal conductivity distribution
from numpy import nan
result_optim_single_material = results_optim_single_material["solution"][-1] + mesh.MaterialCF({"slot.*_bundle.*" : 0}, default = nan)
Draw(result_optim_single_material, results_optim_single_material["solution"][-1].space.mesh,
     settings = {"Objects" : {"Wireframe" : False}, "Colormap" : {"ncolors" : 32}},
     filename = "scenes/optim/single_material/sigma_optim_single_material.html",
     min = sigma_min, max = sigma_max
     )

In [ ]:
# Display result
from utils.physics import average_property
sigma_opt = average_property(results_optim_single_material["solution"][-1], 
                                    result_AC, 
                                    zone = "slot.*_bundle.*")
print(f"Optimal global conductivity = {sigma_opt :.2e} S/m")
print(f"=> optimal Joule losses = {results_optim_single_material['objective'][-1]:.2e} W/m")
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(results_optim_single_material["objective"], 'b-')
ax2.semilogy(results_optim_single_material["criterion"], 'r-')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Joule losses (W/m)', color='b')
ax2.set_ylabel('Stop criterion', color='r')
plt.title(f"Convergence (single material optimization)")
plt.savefig("scenes/optim/single_material/convergence_optim_single_material.pdf")
plt.show()

___
## 4) Post processing and analysis

### 4.a) Torque computation

In [ ]:
from utils.physics import average_torque

torque_optim = average_torque(results_optim_single_material["state"])
print(f"Torque after optimization = {torque_optim/1000:.2f} kNm/m")
print(f"Reference torque = 46.8 kNm/m")

### 4.b) DC analysis

In [ ]:
results_optim_single_material_DC = solve_magnetoharmonic(fes=fes,
                                                         reluctivity=nu,
                                                         conductivity=results_optim_single_material["solution"][-1],
                                                         magnetization=Mcplx,
                                                         frequency=freq/1e6,
                                                         supply=bundles,
                                                         )

total_losses_optim = results_optim_single_material["objective"][-1]
DC_losses_optim = joule_losses(results_optim_single_material_DC)
AC_losses_optim = total_losses_optim - DC_losses_optim

print(f"Optimized total Joule losses (sigma = {sigma_opt:.2e} S/m): {total_losses_optim:.2e} W/m")
print(f"Optimized DC Joule losses (sigma = {sigma_opt:.2e} S/m): {DC_losses_optim:.2e} W/m ({DC_losses_optim/total_losses_optim*100:.1f} %)")
print(f"Optimized AC Joule losses (sigma = {sigma_opt:.2e} S/m): {AC_losses_optim:.2e} W/m ({AC_losses_optim/total_losses_optim*100:.1f} %)")

___
## Summary

We find the following results after a gradient descent at $f = 1\;\text{kHz}$, for a single material optimization (constant condutivity in each conductors):
- $\sigma_{opt} = 8.69\times 10^6 S/m$


| Total losses at 1kHz | $P_{single} = 1.34\times 10^4 \; \text{W/m}$ | 100% |
|---|---|------|
| DC losses | $P_{DC-single} = 6.72\times 10^3  \; \text{W/m}$ | 50 %   |
| AC losses | $P_{AC-single} = 6.72\times 10^3  \; \text{W/m}$ | 50 %   |

It looks more optimal! However, it might not be always a perfect 50/50.

- The torque is $T_{single} = 47.8 \;\text{kNm/m}$, which is ***even better*** than the reference $T_{ref} = 46.8\;\text{kNm/m}$. This is somehow expected since the same current is applied (amplitude and angle) with fewer losses.


We can do even better by relaxing some assumption to extend our admissibility space for the conductivity, which might be possible with modern manufacturing technologies. For instance:
- [4_multi_material_optimization](4_multi_material_optimization.ipynb) (optimization of a different conductivity for each bundle, possible with individualized hairpin winding)
- [5_topology_optimization](5_topology_optimization.ipynb) (optimization of the conductivity at each point of the cross-section, so actually the cross-section itself, possible with additive manufacturing)